In [1]:
import time
import pandas as pd
from zhinst.toolkit import Session
import numpy as np
import matplotlib.pyplot as plt

session = Session("localhost", hf2=True)
device = session.connect_device("DEV1004")

In [2]:
ogFreq = 3347620.0000000843

In [3]:

def get_noise_data(device,  TcSet = 1,   numSamples = 100, demod = 3, tmeasure = np.inf, FreqSet = ogFreq,):
    device.demods[demod].timeconstant(TcSet)
    device.oscs[1].freq(FreqSet)
    TcReal = device.demods[demod].timeconstant()
    FreqReal = device.oscs[1].freq()
    
    Tstart = time.time()
    data = pd.DataFrame()#device.demods[3].sample() )
    # concatenate the data with data
    for i in range(numSamples):
        time.sleep(10*TcSet)
        sample_data = pd.DataFrame(device.demods[demod].sample())
        sample_data['Time'] = time.time() - Tstart
        sample_data['TcSet'] = TcSet
        sample_data['TcReal'] = TcReal
        data = pd.concat([data, sample_data], axis=0)

        if float(sample_data['Time']) > tmeasure:
            break

    return data

In [4]:
data = get_noise_data(device,  TcSet = 0.001, numSamples = 3, demod = 3)
data

,timestamp,x,y,frequency,phase,dio,trigger,auxin0,auxin1,Time,TcSet,TcReal
0,216460665356288,0.000002,-1.252348e-06,3347620.0,1.812426,526385151,3,-0.008240,-0.002136,0.033000,0.001,0.001002
0,216460676562944,-0.000003,5.778937e-07,3347620.0,0.787903,526385151,3,-0.009155,-0.003357,0.078021,0.001,0.001002
0,216460684034048,-0.000001,-8.935857e-07,3347620.0,2.199379,526385151,3,-0.009155,-0.003052,0.125020,0.001,0.001002


In [5]:
results = dict()
Freqs =  5* 10 ** np.linspace(-1, 7, 81)
# for freq in Freqs:
#     time.sleep(1)
#     device.oscs[1].freq(freq)

for Freq in Freqs:
    print(Freq)
    data = get_noise_data(device,  TcSet = 0.1, numSamples = 100000000000000, demod = 3, tmeasure = 1000, FreqSet = Freq)
    results[Freq] = data




0.5
0.6294627058970836
0.7924465962305567
0.9976311574844399
1.25594321575479
1.5811388300841898
1.9905358527674866
2.505936168136362
3.1547867224009667
3.9716411736214075
5.0
6.294627058970837
7.92446596230557
9.976311574844399
12.559432157547905
15.811388300841898
19.905358527674867
25.059361681363622
31.547867224009668
39.716411736214084
50.0
62.94627058970838
79.24465962305571
99.76311574844404
125.5943215754791
158.11388300841895
199.05358527674866
250.59361681363626
315.47867224009684
397.16411736214104
500.0
629.4627058970838
792.4465962305571
997.6311574844403
1255.9432157547913
1581.1388300841897
1990.5358527674866
2505.9361681363625
3154.7867224009683
3971.641173621411
5000.0
6294.627058970844
7924.46596230557
9976.311574844394
12559.43215754791
15811.388300841898
19905.358527674885
25059.361681363625
31547.867224009715
39716.41173621411
50000.0
62946.27058970844
79244.6596230557
99763.11574844414
125594.32157547912
158113.88300841895
199053.58527674887
250593.61681363627
315

In [10]:
results.keys()

dict_keys([0.5, 0.6294627058970836, 0.7924465962305567, 0.9976311574844399, 1.25594321575479, 1.5811388300841898, 1.9905358527674866, 2.505936168136362, 3.1547867224009667, 3.9716411736214075, 5.0, 6.294627058970837, 7.92446596230557, 9.976311574844399, 12.559432157547905, 15.811388300841898, 19.905358527674867, 25.059361681363622, 31.547867224009668, 39.716411736214084, 50.0, 62.94627058970838, 79.24465962305571, 99.76311574844404, 125.5943215754791, 158.11388300841895, 199.05358527674866, 250.59361681363626, 315.47867224009684, 397.16411736214104, 500.0, 629.4627058970838, 792.4465962305571, 997.6311574844403, 1255.9432157547913, 1581.1388300841897, 1990.5358527674866, 2505.9361681363625, 3154.7867224009683, 3971.641173621411, 5000.0, 6294.627058970844, 7924.46596230557, 9976.311574844394, 12559.43215754791, 15811.388300841898, 19905.358527674885, 25059.361681363625, 31547.867224009715, 39716.41173621411, 50000.0, 62946.27058970844, 79244.6596230557, 99763.11574844414, 125594.3215754

In [25]:
# for Freq , data in results.items():
#     print(Freq), data.x.std())

%matplotlib

TcSet = [ data.TcSet.mean() for data in results.values()]
TcReal = [ data.TcReal.mean() for data in results.values()]
x1 = [ data.x.mean() for data in results.values()]
y1 = [ data.y.mean() for data in results.values()]
dx1 = [ data.x.std() for data in results.values()]
dy1 = [ data.y.std() for data in results.values()]

df = pd.DataFrame({'Freqs':Freqs,'TcSet': TcSet, 'TcReal': TcReal, 'x1': x1, 'y1': y1, 'dx1': dx1, 'dy1': dy1})
df["xSensitivity"] = df.dx1 * np.sqrt(df.TcReal)
df["ySensitivity"] = df.dy1 * np.sqrt(df.TcReal)


Using matplotlib backend: <object object at 0x000001A4FF2164B0>


<AxesSubplot: xlabel='Freqs'>

: 

In [ ]:

plt.figure()


df.plot(x='Freqs', y=['dx1','dy1'], logy=True,logx=True,marker='x')

# vertical line at 60Hz
plt.axvline(x=60, color='r', linestyle='--')


plt.figure()
df.plot(x='Freqs', y=['xSensitivity','ySensitivity'], logy=True,logx=True)


In [ ]:
df = pd.read_csv('darkNoise_OscFreqSweep_scantime1000s.csv')

In [23]:
# df.to_csv('darkNoise_OscFreqSweep_scantime1000s.csv')
# results_df.to_csv('darkNoise_OscFreqSweep_scantime1000s_rawdata.csv')

In [21]:
# import pandas as pd

# # Sample DataFrames in a dictionary
# results = {
#     'experiment1': pd.DataFrame({
#         'A': [1, 2, 3],
#         'B': [4, 5, 6]
#     }),
#     'experiment2': pd.DataFrame({
#         'A': [7, 8, 9],
#         'B': [10, 11, 12]
#     })
# }

# Concatenating DataFrames including the dictionary keys as a new column
results_df = pd.concat(results.values(), keys=results.keys(), axis=0).reset_index()
# results_df.rename(columns={'level_0': 'Experiment'}, inplace=True)

# Show the DataFrame
results_df.level_0.unique()

# Uncomment the below line when finalizing the code
# results_df


array([5.00000000e-01, 6.29462706e-01, 7.92446596e-01, 9.97631157e-01,
       1.25594322e+00, 1.58113883e+00, 1.99053585e+00, 2.50593617e+00,
       3.15478672e+00, 3.97164117e+00, 5.00000000e+00, 6.29462706e+00,
       7.92446596e+00, 9.97631157e+00, 1.25594322e+01, 1.58113883e+01,
       1.99053585e+01, 2.50593617e+01, 3.15478672e+01, 3.97164117e+01,
       5.00000000e+01, 6.29462706e+01, 7.92446596e+01, 9.97631157e+01,
       1.25594322e+02, 1.58113883e+02, 1.99053585e+02, 2.50593617e+02,
       3.15478672e+02, 3.97164117e+02, 5.00000000e+02, 6.29462706e+02,
       7.92446596e+02, 9.97631157e+02, 1.25594322e+03, 1.58113883e+03,
       1.99053585e+03, 2.50593617e+03, 3.15478672e+03, 3.97164117e+03,
       5.00000000e+03, 6.29462706e+03, 7.92446596e+03, 9.97631157e+03,
       1.25594322e+04, 1.58113883e+04, 1.99053585e+04, 2.50593617e+04,
       3.15478672e+04, 3.97164117e+04, 5.00000000e+04, 6.29462706e+04,
       7.92446596e+04, 9.97631157e+04, 1.25594322e+05, 1.58113883e+05,
      